In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_pth=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)
from utils.transformation import reusable

In [0]:
df1=spark.read.format("parquet")\
    .load("abfss://bronze@spotifyprojectazdb.dfs.core.windows.net/DimUser")

In [0]:
display(df1)

***'Autoloader'***

In [0]:

df_user=spark.readStream.format("CloudFiles")\
.option("CloudFiles.format","parquet")\
.option("CloudFiles.schemaLocation","abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/Checkpoint")\
.option("schemaEvolutionMode","rescue")\
.load("abfss://bronze@spotifyprojectazdb.dfs.core.windows.net/DimUser")

In [0]:
display(
    df_user,
    checkpointLocation="abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/Checkpoint/display"
)

In [0]:
df_user=df_user.withColumn("user_name",upper(col("user_name")))
display(df_user)

In [0]:
df_user_obj=reusable()
df_user=df_user_obj.dropcolums(df_user,['_rescued_data'])
df_user=df_user.dropDuplicates(df_user,['user_id'])
df_user_clean = df_user.drop("_rescued_data")
(df_user_clean.writeStream
    .format("Delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/checkpoint/write")
    .trigger(once=True)
    .start("abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/data"))

In [0]:
df_user_obj=reusable()
df_user=df_user_obj.dropcolums(df_user,['_rescued_data'])
df_user=df_user.dropDuplicates(['user_id'])
df_user_clean = df_user.drop("_rescued_data")
query = (df_user_clean.writeStream
    .format("memory")
    .queryName("dimuser_preview")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/checkpoint/dimuser_preview")
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

display(spark.sql("SELECT * FROM dimuser_preview"))

In [0]:
df_user.writeStream.format("Delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .start("abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/data")

In [0]:
(df_user.writeStream.format("Delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser"))

** DIMARTIST**

In [0]:
df_artist=spark.readStream.format("CloudFiles")\
.option("CloudFiles.format","parquet")\
.option("CloudFiles.schemaLocation","abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimArtist/Checkpoint")\
.option("schemaEvolutionMode","rescue")\
.load("abfss://bronze@spotifyprojectazdb.dfs.core.windows.net/DimArtist")

In [0]:
display(
    df_artist,
    checkpointLocation="abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimArtist/Checkpoint/display"
)

In [0]:
df_artist_obj=reusable()
df_art=df_artist_obj.dropcolums(df_artist,['_rescued_data'])
df_art=df_art.dropDuplicates(['artist_id'])
(df_art.writeStream
    .format("Delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimArtist/checkpoint")
    .trigger(once=True)
    .start("abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimArtist/data"))

In [0]:


(df_art.writeStream.format("Delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist"))


DIMTRACK

In [0]:
df_track=spark.readStream.format("CloudFiles")\
.option("CloudFiles.format","parquet")\
.option("CloudFiles.schemaLocation","abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/Checkpoint")\
.option("schemaEvolutionMode","rescue")\
.load("abfss://bronze@spotifyprojectazdb.dfs.core.windows.net/DimTrack")

In [0]:
display(
    df_track,
    checkpointLocation="abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/Checkpoint/display"
)

In [0]:
df_track_obj=reusable()
df_track=df_track_obj.dropcolums(df_track,['_rescued_data'])
df_track=df_track.dropDuplicates(['track_id'])
(df_track.writeStream
    .format("Delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/checkpoint")
    .trigger(once=True)
    .start("abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/data"))

In [0]:
(df_track.writeStream.format("Delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(availableNow=True)\
    .option("path","abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.Dimtrack"))

In [0]:
(df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/checkpoint_v2")
    .trigger(availableNow=True)
    .option("path", "abfss://silver@spotifyprojectazdb.dfs.core.windows.net/DimTrack/data_v2")
    .toTable("spotify_cata.silver.DimTrack_v2"))

In [0]:
df_track=df_track.withColumn("duraionFlag",when(col('duration_sec')<150,"low")
                             \.otherwi)